In [1]:
import ollama

In [2]:
models_dict = ollama.list()

In [3]:
model_name = models_dict.models[11].model #gemma3:4b
print(model_name)

gemma3:4b


In [4]:
# Célula 2: Importação das bibliotecas e configuração do pipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_192994/232072495.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
# ==========================================
# 1. CARREGAMENTO E DIVISÃO DO PDF
# ==========================================
# Substitua pelo caminho real do seu arquivo PDF no computador
caminho_pdf = "Regulamento_Estagio_UFRB.pdf" 

print(f"Carregando o arquivo: {caminho_pdf}...")
loader = PyPDFLoader(caminho_pdf)
documentos_carregados = loader.load()


Carregando o arquivo: Regulamento_Estagio_UFRB.pdf...


In [7]:

# Como PDFs podem ser gigantescos, dividimos o texto em blocos (chunks) menores
# chunk_size: tamanho do bloco | chunk_overlap: sobreposição para não cortar contextos no meio
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documentos_fragmentados = text_splitter.split_documents(documentos_carregados)
print(f"PDF dividido em {len(documentos_fragmentados)} blocos de texto.")



PDF dividido em 71 blocos de texto.


In [8]:

# ==========================================
# 2. EMBEDDINGS E BANCO DE VETORES
# ==========================================
diretorio_db = "./data_documents"
# Criando IDs únicos baseados no nome do arquivo e número do pedaço


embeddings = OllamaEmbeddings(model="nomic-embed-text")
ids = [f"doc_{caminho_pdf.split('.')[0]}_chunk_{i}" for i in range(len(documentos_fragmentados))]

print("Indexando blocos do PDF no banco de vetores local...")
# Criamos o banco a partir dos documentos fragmentados extraídos do PDF
vectorstore = Chroma.from_documents(
    documents=documentos_fragmentados, 
    embedding=embeddings,
    persist_directory=diretorio_db,
    ids=ids
)

#preparação para a pergunta com similaridade de 3 trechos mais relevantes parecidos
# Configura o buscador para trazer os 3 blocos mais relevantes para a pergunta
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Indexando blocos do PDF no banco de vetores local...


In [10]:

# ==========================================
# 3. CONFIGURAÇÃO DO LLM E PROMPT RAG
# ==========================================

# temperature indica o nível de criatividade da LLM, ou grau de liberdade
llm = ChatOllama(model=model_name, temperature=0)

template_rag = """
[CONTEXTO]
Utilize estritamente as informações extraídas do regulamento de estágio da UFRB em PDF fornecidas abaixo para responder à pergunta do usuário de forma clara e direta. 
Se a resposta não puder ser encontrada nos fragmentos, diga apenas que a informação não consta no documento.

FRAGMENTOS DO PDF:
{context}

[PERGUNTA]
{question}

[RESPOSTA]
"""
prompt = ChatPromptTemplate.from_template(template_rag)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construção da cadeia RAG
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Sistema de RAG com suporte a PDF pronto!")

Sistema de RAG com suporte a PDF pronto!


In [11]:
# Célula 3: Consulta ao documento PDF
pergunta = "Quais documentos são necessários para redução de carga horária?"

# O sistema busca no PDF, monta o contexto e o Ollama responde localmente
resposta_final = rag_chain.invoke(pergunta)

print(resposta_final)

Para efeito de redução de carga horária de Estágio Curricular Supervisionado, o discente deverá apresentar:

I - Comprovante de vínculo empregatício (cópia da Carteira de Trabalho ou cópia de nomeação no Diário Oficial);
II - Três últimos contracheques (apenas a parte que indica nome, matrícula e mês do pagamento);
III - Atestado de frequência da escola, discriminando nível de ensino, ano, disciplina, turno e carga horária;
IV - Relatório da Coordenação de Área, ou Coordenação Pedagógica ou da Direção, avaliando o perfil profissional do professor em formação.


In [12]:
# Célula 3: Consulta ao documento PDF
pergunta = "Pode ser solicitado dispensa do estágio supervisionado obrigatório?"

# O sistema busca no PDF, monta o contexto e o Ollama responde localmente
resposta_final = rag_chain.invoke(pergunta)

print(resposta_final)

A informação não consta no documento.


# Atividade
Faça um complemento a esse notebook, iterando os modelos gemma3:12b, llama3.2-vision:11b e qwen3-vl:30b para a pergunta acima, verificar qual obteve a melhor resposta. Modularize o código de modo que encapsule as funcionalidades